# Anomaly Detection

Anomaly Detection identifies observations that are unusual compared with normal observations.

## Workflow

**Load → EDA → Preprocessing → Feature Engineering → Select Features → Handle Missing Values/Outliers → Scale Data → Train Anomaly Detector → Predict Anomalies → Evaluate → Compare → Improve**

Anomaly Detection can be:
- **Unsupervised** → no anomaly labels
- **Semi-supervised** → mostly normal data, few/no anomaly labels
- **Supervised** → anomaly labels are available

Load dataset

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("data.csv")

df.head()

EDA

In [ ]:
print("Shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicates:", df.duplicated().sum())

print("\nData Types:")
print(df.dtypes)

In [ ]:
df.describe(include="all")

### Preprocessing

Main preprocessing:
- Remove duplicates
- Handle missing values
- Select useful numerical features
- Encode categorical variables if required
- Scale features when the algorithm is distance/geometry based
- Create domain-specific features
- Do not automatically remove outliers before anomaly detection

**Important:** The observations you want to detect as anomalies may be the very observations you must preserve.

Select Features

In [ ]:
# Replace these with the features relevant to your problem.

X = df.select_dtypes(
    include=["int64", "float64"]
).copy()

print("Selected features:")
print(X.columns.tolist())

Handle Missing Values

In [ ]:
X = X.fillna(X.median())

print("Remaining missing values:",
      X.isnull().sum().sum())

Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print("Shape:", X_scaled.shape)

### Train / Test Split

For unsupervised anomaly detection, there is normally **no standard train/test split with a target `y`**.

If labeled anomalies are available, use a chronological or stratified train/test split depending on the problem.

If the data is time-based, split by time rather than randomly.

### Evaluation

If true anomaly labels are available:

- **Precision** → higher is better
- **Recall** → higher is better
- **F1** → higher is better
- **PR-AUC** → useful when anomalies are rare
- **ROC-AUC** → can be used when scores and labels are available

If labels are NOT available, there is no direct Accuracy/Precision/Recall.

Use:
- Number/percentage of detected anomalies
- Stability across parameter settings
- Score distributions
- Domain validation / manual inspection

**The best detector is not simply the one that flags the most points.**

# Z-Score

**Use:** Simple statistical detection when a feature is approximately normally distributed and extreme values are the main concern.

Typical rule:

`|z-score| > threshold → anomaly`

This is a statistical baseline, not a machine-learning model.

In [ ]:
from scipy.stats import zscore

Z = np.abs(zscore(X))

threshold = 3

anomaly_mask_z = (Z > threshold).any(axis=1)

print("Anomalies:", anomaly_mask_z.sum())
print("Anomaly %:", round(anomaly_mask_z.mean() * 100, 2))

# IQR Method

**Use:** Simple rule-based detection when extreme values can be identified using the interquartile range.

Typical rule:

`value < Q1 - 1.5×IQR` or `value > Q3 + 1.5×IQR` → anomaly

In [ ]:
Q1 = X.quantile(0.25)
Q3 = X.quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

anomaly_mask_iqr = ((X < lower) | (X > upper)).any(axis=1)

print("Anomalies:", anomaly_mask_iqr.sum())
print("Anomaly %:", round(anomaly_mask_iqr.mean() * 100, 2))

# Isolation Forest

**Use:** Strong general-purpose anomaly detection for tabular data when anomalies are relatively rare.

Main parameters:
- `n_estimators`
- `contamination`
- `max_samples`

Output:
- `1` → normal
- `-1` → anomaly

In [ ]:
from sklearn.ensemble import IsolationForest

model_if = IsolationForest(
    n_estimators=200,
    contamination="auto",
    random_state=42
)

labels_if = model_if.fit_predict(X_scaled)

anomaly_mask_if = labels_if == -1

print("Anomalies:", anomaly_mask_if.sum())
print("Anomaly %:", round(anomaly_mask_if.mean() * 100, 2))

# Local Outlier Factor (LOF)

**Use:** When anomalies are local to a neighborhood and normal observations can have different local densities.

Main parameters:
- `n_neighbors`
- `contamination`

LOF is useful when global distance is less informative than local density.

In [ ]:
from sklearn.neighbors import LocalOutlierFactor

model_lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination="auto"
)

labels_lof = model_lof.fit_predict(X_scaled)

anomaly_mask_lof = labels_lof == -1

print("Anomalies:", anomaly_mask_lof.sum())
print("Anomaly %:", round(anomaly_mask_lof.mean() * 100, 2))

# One-Class SVM

**Use:** When you mainly have normal examples and want to learn a boundary around normal observations.

Main parameters:
- `nu`
- `kernel`
- `gamma`

Scaling is important.

In [ ]:
from sklearn.svm import OneClassSVM

model_ocsvm = OneClassSVM(
    kernel="rbf",
    nu=0.05,
    gamma="scale"
)

labels_ocsvm = model_ocsvm.fit_predict(X_scaled)

anomaly_mask_ocsvm = labels_ocsvm == -1

print("Anomalies:", anomaly_mask_ocsvm.sum())
print("Anomaly %:", round(anomaly_mask_ocsvm.mean() * 100, 2))

# Elliptic Envelope

**Use:** When normal observations are approximately Gaussian/elliptical and you want to detect observations outside that normal region.

Main parameter:
- `contamination`

Works best when the normal data has a reasonably elliptical distribution.

In [ ]:
from sklearn.covariance import EllipticEnvelope

model_ee = EllipticEnvelope(
    contamination=0.05,
    random_state=42
)

labels_ee = model_ee.fit_predict(X_scaled)

anomaly_mask_ee = labels_ee == -1

print("Anomalies:", anomaly_mask_ee.sum())
print("Anomaly %:", round(anomaly_mask_ee.mean() * 100, 2))

# Model Comparison

If anomaly labels are available, compare models using:

**Precision / Recall / F1 / PR-AUC**

If labels are unavailable, compare:

**Detected anomaly percentage + score distribution + stability + domain validation**

Do not compare detectors only by how many anomalies they produce.

In [ ]:
# Example comparison when true anomaly labels are available.
# y_true should contain:
# 0 = normal
# 1 = anomaly

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

def evaluate_anomaly(name, y_true, predicted_labels):
    return {
        "Model": name,
        "Precision": precision_score(
            y_true, predicted_labels, zero_division=0
        ),
        "Recall": recall_score(
            y_true, predicted_labels, zero_division=0
        ),
        "F1": f1_score(
            y_true, predicted_labels, zero_division=0
        )
    }

# Convert detector output:
# -1 = anomaly → 1
#  1 = normal  → 0

# pred_if = (labels_if == -1).astype(int)
# pred_lof = (labels_lof == -1).astype(int)
# pred_ocsvm = (labels_ocsvm == -1).astype(int)
# pred_ee = (labels_ee == -1).astype(int)

# results = pd.DataFrame([
#     evaluate_anomaly("Isolation Forest", y_true, pred_if),
#     evaluate_anomaly("LOF", y_true, pred_lof),
#     evaluate_anomaly("One-Class SVM", y_true, pred_ocsvm),
#     evaluate_anomaly("Elliptic Envelope", y_true, pred_ee)
# ])
#
# results.sort_values("F1", ascending=False)

In [ ]:
# Unsupervised comparison when labels are NOT available

unsupervised_comparison = pd.DataFrame([
    {
        "Model": "Z-Score",
        "Anomalies": anomaly_mask_z.sum(),
        "Anomaly %": anomaly_mask_z.mean() * 100
    },
    {
        "Model": "IQR",
        "Anomalies": anomaly_mask_iqr.sum(),
        "Anomaly %": anomaly_mask_iqr.mean() * 100
    },
    {
        "Model": "Isolation Forest",
        "Anomalies": anomaly_mask_if.sum(),
        "Anomaly %": anomaly_mask_if.mean() * 100
    },
    {
        "Model": "LOF",
        "Anomalies": anomaly_mask_lof.sum(),
        "Anomaly %": anomaly_mask_lof.mean() * 100
    },
    {
        "Model": "One-Class SVM",
        "Anomalies": anomaly_mask_ocsvm.sum(),
        "Anomaly %": anomaly_mask_ocsvm.mean() * 100
    },
    {
        "Model": "Elliptic Envelope",
        "Anomalies": anomaly_mask_ee.sum(),
        "Anomaly %": anomaly_mask_ee.mean() * 100
    }
])

unsupervised_comparison

# Model Improvement

**Baseline → Better Features → Choose suitable detector → Tune parameters → Validate anomaly rate → Domain validation → Final detector**

### Feature Engineering
- Create domain-specific features
- Add ratios or differences
- Add rolling statistics for time-series data
- Remove irrelevant features
- Transform highly skewed variables when justified

### Isolation Forest
Tune:
- `n_estimators`
- `contamination`
- `max_samples`
- `max_features`

### LOF
Tune:
- `n_neighbors`
- `contamination`
- `metric`

### One-Class SVM
Tune:
- `nu`
- `gamma`
- `kernel`

### Elliptic Envelope
Tune:
- `contamination`
- `support_fraction`

### Threshold Tuning
Instead of blindly using the default anomaly threshold, choose a threshold based on:
- Business cost of false positives
- Business cost of false negatives
- Available labeled validation data
- Desired alert volume

### Important

If anomalies are rare, **Recall and Precision can matter very differently**.

For example:
- Fraud detection may prioritize Recall
- Expensive manual investigations may prioritize Precision

# Time-Series Anomaly Detection

For time-series data, add temporal features before applying many general anomaly detectors.

Examples:
- Lag values
- Rolling mean
- Rolling standard deviation
- Difference from previous period
- Seasonal difference
- Trend
- Day/week/month features

**Do not use future values to calculate features for the current observation.**

For forecasting-based anomaly detection, another approach is:

**Forecast → Calculate residual → Large residual → Anomaly**

In [ ]:
# Example residual-based idea

# forecast = model.predict(X_test)

# residual = y_test - forecast
# threshold = residual.abs().quantile(0.99)

# anomalies = residual.abs() > threshold

# print("Anomalies:", anomalies.sum())

# Quick Revision

**Z-Score** → simple statistical baseline; extreme standardized values

**IQR** → simple statistical baseline; quartile-based outliers

**Isolation Forest** → isolates unusual observations; strong general-purpose choice

**LOF** → detects local density-based outliers

**One-Class SVM** → learns a boundary around mostly normal data

**Elliptic Envelope** → assumes normal data has an approximately Gaussian/elliptical shape

### Main selection

**Simple numeric outliers → Z-Score / IQR**

**General tabular anomaly detection → Isolation Forest**

**Local-density anomalies → LOF**

**Mostly normal training data → One-Class SVM**

**Approximately Gaussian normal data → Elliptic Envelope**

### Main workflow difference

Regression/Classification:
**X + y → Train/Test → Predict → target metrics**

Clustering:
**X only → Scale → Discover groups → clustering metrics**

Anomaly Detection:
**X → Preprocess → Detect unusual observations → anomaly scores/labels → validate with labels or domain knowledge**